# 04 — Federated FedAvg baseline (Phase 3)

Simulated hospitals (Dirichlet ward-mixture) train together via **FedAvg** (Flower).
Compares **pooled** (Phase 1) vs **local-only** (each hospital alone) vs **FedAvg**.

**To get new code after a `git pull`:** just re-run cell 1 (pull) then the run cell —
cells 4/5 force-reload `amr_fed` from disk, so **no kernel restart is needed**.
All simulated on ONE machine — no second computer needed.

In [ ]:
# 1) Get the code + deps
!git clone -b phase3-federated https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout phase3-federated && git pull)
!pip install -q torch_geometric 'flwr[simulation]'

In [ ]:
# 2) Point at the data (mount Drive, set ARMD_DIR before importing amr_fed)
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # EDIT to your ARMD folder

In [ ]:
# 3) Sanity: data resolves
import sys
sys.path.insert(0, '/content/Capstone-amr-fed/src')
from amr_fed import config
from pathlib import Path
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong — fix cell 2 and re-run.'

In [ ]:
# 4) Run FedAvg at alpha=0.5 (5 hospitals). Prints local-only vs FedAvg vs pooled.
# Force-reload amr_fed from disk so a `git pull` takes effect WITHOUT a kernel restart.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]

from amr_fed.federated.run import run_fedavg
res = run_fedavg(alpha=0.5, n_clients=5, rounds=10, local_epochs=6)
print(res)

In [ ]:
# 5) MULTI-SEED alpha sweep — mean +/- std to denoise partition + training noise.
# 3 seeds x 3 alphas = 9 FedAvg runs, ~20-40 min. Cohort loaded once and reused.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame

df = load_cohort_frame()
summaries = [run_multiseed(alpha=a, seeds=(42, 43, 44), df=df) for a in (0.1, 0.5, 1.0)]

print("\n=== ALPHA SWEEP (multi-seed, mean +/- std) ===")
for s in summaries:
    print(f"alpha={s['alpha']}: local {s['local_only'][0]}+/-{s['local_only'][1]} | "
          f"FedAvg-best {s['fedavg_best'][0]}+/-{s['fedavg_best'][1]} | "
          f"gain {s['gain_mean']}+/-{s['gain_std']}")

In [ ]:
# 6) NON-IID SPLIT #1: label-Dirichlet on RESISTANT-RATE (the axis FedAvg struggles with).
# Splits patients so hospitals differ in their resistant/susceptible mix. Sweeps beta
# (small = strong label skew). Expect local-only to DROP and the FedAvg gain + worst-
# hospital gain to GROW vs the ward split -- i.e. more room for the topology-aware method.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame
from amr_fed.partition import label_dirichlet

df = load_cohort_frame()
for beta in (0.1, 0.5):
    run_multiseed(seeds=(42, 43, 44), df=df, label=f"label-dir beta={beta}",
                  partition_fn=lambda d, s, b=beta: label_dirichlet(d, n_clients=5, beta=b, seed=s))

In [ ]:
# 7) NON-IID SPLIT #3: natural SPECIMEN split (urine / respiratory / blood / other).
# One hospital per specimen source -- clinically defensible, and resistance varies a lot
# by source (EDA: urine ~0.18 vs resp ~0.29), so this is a real label + topology skew.
# n_clients is derived from the split (3-4 hospitals); seeds vary only training noise.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame
from amr_fed.partition import specimen_baseline

df = load_cohort_frame()
run_multiseed(seeds=(42, 43, 44), df=df, label="specimen",
              partition_fn=lambda d, s: specimen_baseline(d))

In [ ]:
# 8) NON-IID SPLIT #4: ORGANISM-community split -- each hospital sees a DIFFERENT set of
# bugs (maximal structural/topology heterogeneity; the tightest fit for topology-aware
# aggregation). Organisms are greedily packed into 5 balanced hospitals. Expect the
# strongest FedAvg + worst-hospital gains of all the splits.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame
from amr_fed.partition import organism_community

df = load_cohort_frame()
run_multiseed(seeds=(42, 43, 44), df=df, label="organism-community",
              partition_fn=lambda d, s: organism_community(d, n_clients=5, seed=s))

In [ ]:
# 9) HARD SPLIT #1: HOMOPHILY spectrum -- hospitals span heterophilic <-> homophilic.
# Calibrated protocol: 8 hospitals, 6 rounds x 3 local epochs = 18 budgets.
# resistance rate is a reported covariate (no longer held constant).
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame
from amr_fed.partition import homophily_split

df = load_cohort_frame()
run_multiseed(seeds=(42, 43, 44), df=df, label="homophily",
              partition_fn=homophily_split, n_clients=8, rounds=6,
              local_epochs=3, local_only_epochs=18, hidden=64)

In [ ]:
# 10) HARD SPLIT #2: DEGREE SKEW -- hospitals span sparse <-> hub-heavy (drug-repertoire breadth).
# De-saturated hubness score (log1p distinct antibiotics); calibrated protocol.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame
from amr_fed.partition import degree_skew_split

df = load_cohort_frame()
run_multiseed(seeds=(42, 43, 44), df=df, label="degree-skew",
              partition_fn=degree_skew_split, n_clients=8, rounds=6,
              local_epochs=3, local_only_epochs=18, hidden=64)

In [ ]:
# 11) HARD SPLIT #3: TOPOLOGY quadrant -- crossed homophily x degree, purity=0.0.
# Sixteen hospitals = 4 corners x 2 buckets (n_clients must be a multiple of 4).
# FedGTA-style 2-D structure non-IID partition. decorrelate=False for the first run
# (the _self_check diagnostic will tell us whether to flip it).
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from functools import partial
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame
from amr_fed.partition import topology_split

df = load_cohort_frame()
run_multiseed(seeds=(42, 43, 44), df=df, label="topology-corners",
              partition_fn=partial(topology_split, purity=0.0, decorrelate=False),
              n_clients=8, rounds=6, local_epochs=3, local_only_epochs=18, hidden=64)

In [ ]:
# 12) HARD SPLIT #4: LOUVAIN communities -- FedGTA-style topology split.
# Hospitals built from Louvain community detection on the organism--antibiotic
# test graph, greedily packed. Effective hospital count may be less than
# n_clients if there are fewer communities; run_fedavg self-heals.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame
from amr_fed.partition import louvain_split

df = load_cohort_frame()
run_multiseed(seeds=(42, 43, 44), df=df, label="louvain-communities",
              partition_fn=louvain_split, n_clients=5, rounds=6,
              local_epochs=3, local_only_epochs=18, hidden=64)

In [ ]:
# 13) HEADROOM GATE -- the acceptance test: does this split actually break FedAvg
# so the Phase-5 topology-aware aggregator has measurable headroom?
#   PASS = FedAvg trail pooled by >=0.02 (worst) or >=0.01 (mean)
#          AND pooled still strong (pooled_worst >= 0.60)
#          AND FedAvg beats local-only
#   If FAIL, read the hint and adjust purity / hidden / rounds / n_clients.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from functools import partial
from amr_fed.federated.run import headroom_gate
from amr_fed.data_loader import load_cohort_frame
from amr_fed.partition import topology_split, louvain_split

df = load_cohort_frame()
# Try topology_split first -- the strongest structure-only non-IID according to
# FedGTA + AdaFGL. Adjust once at a time using the hints.
res = headroom_gate(partition_fn=partial(topology_split, purity=0.0),
                    n_clients=8, rounds=6, local_epochs=3, hidden=64,
                    df=df, label="topology-corners")
#   fed_helps fails -> raise purity (0.2) / rounds (8) / local_epochs (4)
#   gap too small   -> lower purity to 0.0, try hidden=32, or rounds=4
#   pooled weak     -> widen hidden to 128, reduce n_clients to 4
#   When PASS, record the winning config in docs.